# Marginal versus joint smoothing

This compact tutorial fits the same dynamic diffusion model with Superstats' two smoothing approximations. The marginal approximator estimates each $v_t$ conditional on the observations and sampled invariants. The joint approximator additionally conditions each $v_t$ on the preceding trajectory.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

import bayesflow as bf
import superstats as sup

INFO:bayesflow:Multiple Keras-compatible backends detected (JAX, PyTorch, TensorFlow). Defaulting to JAX.
To override, set the KERAS_BACKEND environment variable before importing bayesflow.
See: https://keras.io/getting_started/#configuring-your-backend
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
INFO:bayesflow:Using backend 'jax'


In [ ]:
NUM_STEPS = 100
TRAIN_SIZE = 20_000
VALIDATION_SIZE = 250
TEST_SIZE = 250
EPOCHS = 25
BATCH_SIZE = 32
NUM_SAMPLES = 250

## Model and shared simulations

In [3]:
joint_prior = sup.JointPrior(
    v=sup.transition.RandomWalk(bounds=(-6.0, 6.0)),
    a=sup.transition.Linear(
        bounds=(0.2, 4.0),
        intercept=sup.Prior("normal", loc=2.0, scale=0.5),
        slope=sup.Prior("normal", loc=0.0, scale=1.0),
    ),
    tau=sup.Prior("halfnormal", scale=0.5),
    bias=0.5,
)

model = sup.Model(
    prior=joint_prior,
    simulator=sup.simulation.sample_ddm,
    missing=None,
    contamination=None,
)

/home/radevs/anaconda3/envs/bfjax/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [4]:
train_data = model.sample(batch_size=TRAIN_SIZE, num_steps=NUM_STEPS)
validation_data = model.sample(batch_size=VALIDATION_SIZE, num_steps=NUM_STEPS)
test_data = model.sample(batch_size=TEST_SIZE, num_steps=NUM_STEPS)

## Approximators

The string interface selects the approximation family. Both use Superstats' recurrent summary network and depth-2 spline coupling flows. Joint smoothing passes the shared summaries to BayesFlow's autoregressive decoder.

In [5]:
marginal_workflow = sup.Workflow(model=model, approximator="marginal", mode="smoothing")

joint_workflow = sup.Workflow(model=model, approximator="joint", mode="smoothing")

workflows = {"Marginal": marginal_workflow, "Joint": joint_workflow}

## Train and sample

Both workflows see the same simulations for the same number of optimizer updates. BayesFlow applies its default warm-started cosine-decay learning-rate schedule.

In [ ]:
histories = {}
for name, workflow in workflows.items():

    print(f"Training {name.lower()} smoother")
    histories[name] = workflow.fit_offline(
        data=train_data,
        validation_data=validation_data,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        save_history=False
    )

INFO:bayesflow:Fitting on dataset instance of OfflineDataset.
INFO:bayesflow:Building on a test batch.


Training marginal smoother
Epoch 1/50
625/625 - 25s - 40ms/step - invariant/loss: 4.2684 - loss: 4.6269 - val_invariant/loss: 4.3277 - val_loss: 4.5938 - val_varying/loss: 0.2661 - varying/loss: 0.3584
Epoch 2/50
625/625 - 10s - 17ms/step - invariant/loss: 3.4990 - loss: 3.4487 - val_invariant/loss: 4.6532 - val_loss: 4.6323 - val_varying/loss: -2.0953e-02 - varying/loss: -5.0294e-02
Epoch 3/50
625/625 - 10s - 17ms/step - invariant/loss: 3.4996 - loss: 3.4797 - val_invariant/loss: 3.2923 - val_loss: 3.1526 - val_varying/loss: -1.3968e-01 - varying/loss: -1.9853e-02
Epoch 4/50
625/625 - 10s - 17ms/step - invariant/loss: 2.0738 - loss: 1.7057 - val_invariant/loss: 2.2797 - val_loss: 1.9571 - val_varying/loss: -3.2266e-01 - varying/loss: -3.6804e-01
Epoch 5/50
625/625 - 11s - 17ms/step - invariant/loss: 1.0828 - loss: 0.6073 - val_invariant/loss: 1.1294 - val_loss: 0.8692 - val_varying/loss: -2.6025e-01 - varying/loss: -4.7544e-01
Epoch 6/50
625/625 - 11s - 17ms/step - invariant/loss: 1.6

INFO:bayesflow:Training completed in 10.45 minutes.
INFO:bayesflow:Fitting on dataset instance of OfflineDataset.
INFO:bayesflow:Building on a test batch.


Training joint smoother
Epoch 1/50
625/625 - 43s - 69ms/step - invariant/loss: 5.3059 - loss: 4.9835 - val_invariant/loss: 5.1096 - val_loss: 4.5301 - val_varying/loss: -5.7945e-01 - varying/loss: -3.2244e-01
Epoch 2/50
625/625 - 15s - 24ms/step - invariant/loss: 3.3294 - loss: 2.7451 - val_invariant/loss: 3.0194 - val_loss: 2.1037 - val_varying/loss: -9.1576e-01 - varying/loss: -5.8436e-01
Epoch 3/50
625/625 - 16s - 25ms/step - invariant/loss: 3.0313 - loss: 2.0835 - val_invariant/loss: 2.3644 - val_loss: 1.5636 - val_varying/loss: -8.0076e-01 - varying/loss: -9.4781e-01
Epoch 4/50
625/625 - 17s - 27ms/step - invariant/loss: 1.9455 - loss: 1.1218 - val_invariant/loss: 2.0025 - val_loss: 1.1241 - val_varying/loss: -8.7848e-01 - varying/loss: -8.2366e-01
Epoch 5/50
625/625 - 18s - 29ms/step - invariant/loss: 2.1188 - loss: 1.3312 - val_invariant/loss: 1.8656 - val_loss: 0.9939 - val_varying/loss: -8.7175e-01 - varying/loss: -7.8769e-01
Epoch 6/50
625/625 - 18s - 29ms/step - invariant/lo

KeyboardInterrupt: 

In [8]:
posterior_samples = {
    name: workflow.sample(test_data, num_samples=NUM_SAMPLES, batch_size=4)
    for name, workflow in workflows.items()
}

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

...

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

Sampling:   0%|          | 0/1 [00:00<?, ?batch/s]

RuntimeError: Exception encountered when calling Standardize.call().

[1mArray has been deleted with shape=float32[3].[0m

Arguments received by Standardize.call():
  • x=jnp.ndarray(shape=(4, 100, 3), dtype=float32)
  • stage='inference'
  • forward=True
  • log_det_jac=False
  • transformation_type='location_scale'
  • mask=None

## Time-varying verification

The same held-out trajectories are used for both diagnostic figures.

In [7]:
metric_summary = {}
for name, workflow in workflows.items():

    estimates = posterior_samples[name]["v"]
    targets = test_data["v"]

    metric_summary[name] = {
        "correlation": round(np.mean(sup.diagnostics.correlation_per_step(estimates, targets)), 3),
        "nRMSE": round(np.mean(sup.diagnostics.nrmse_per_step(estimates, targets)), 3),
        "contraction": round(np.mean(sup.diagnostics.posterior_contraction_per_step(estimates, targets)), 3),
        "calibration error": round(np.mean(sup.diagnostics.calibration_error_per_step(estimates, targets)), 3),
    }

    fig = workflow.verify_time_varying(
        targets=test_data,
        estimates=posterior_samples[name],
        variable_keys=["v"]
    )

    fig.suptitle(f"{name} smoothing", y=1.02)

metric_summary

NameError: name 'posterior_samples' is not defined

## One estimated trajectory

Both panels show posterior uncertainty for the first held-out trajectory.

In [ ]:
for name, workflow in workflows.items():
    fig = workflow.plot_time_varying_posterior(
        estimates=posterior_samples[name],
        targets=test_data,
        variable_keys=["v"],
        data_idx=0,
        marginal=False
    )
    fig.suptitle(f"{name} smoothing: one trajectory", y=1.02)